In [1]:
import pandas as pd
import numpy as np

In [2]:
df = pd.read_csv('/home/patel-krish/anaconda3/BrainyBeam/womens_clothing/dataset/womens_clothing.csv')

print("Shape:", df.shape)
print("Exact duplicate rows:", df.duplicated().sum())

near_dup_cols = ['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating']
print(f"Near-duplicates on {near_dup_cols}:", df.duplicated(subset=near_dup_cols).sum())

print("\nColumn Cardinality & Dominance:")
for col in df.columns:
    nunique = df[col].nunique(dropna=False)
    top_ratio = df[col].value_counts(normalize=True, dropna=False).iloc[0]
    print(f"  {col:25s} | nunique={nunique:4d} | unique_ratio={nunique/len(df):.4f} | top_val_ratio={top_ratio:.4f}")

Shape: (1500, 10)
Exact duplicate rows: 0
Near-duplicates on ['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating']: 17

Column Cardinality & Dominance:
  Clothing ID               | nunique=  10 | unique_ratio=0.0067 | top_val_ratio=0.1147
  Age                       | nunique=  67 | unique_ratio=0.0447 | top_val_ratio=0.0273
  Title                     | nunique=   9 | unique_ratio=0.0060 | top_val_ratio=0.1533
  Review Text               | nunique=   5 | unique_ratio=0.0033 | top_val_ratio=0.2553
  Rating                    | nunique=   5 | unique_ratio=0.0033 | top_val_ratio=0.5620
  Recommended IND           | nunique=   2 | unique_ratio=0.0013 | top_val_ratio=0.7847
  Positive Feedback Count   | nunique=  19 | unique_ratio=0.0127 | top_val_ratio=0.2980
  Division Name             | nunique=   3 | unique_ratio=0.0020 | top_val_ratio=0.3367
  Department Name           | nunique=   6 | unique_ratio=0.0040 | top_val_ratio=0.1847
  Class Name                | nunique=  10 | unique_r

In [3]:
num_cols = df.select_dtypes(include=[np.number]).columns
print("Numeric Correlation Matrix:")
print(df[num_cols].corr().round(3))

Numeric Correlation Matrix:
                         Clothing ID    Age  Rating  Recommended IND  \
Clothing ID                    1.000  0.023   0.010           -0.001   
Age                            0.023  1.000  -0.038           -0.034   
Rating                         0.010 -0.038   1.000            0.713   
Recommended IND               -0.001 -0.034   0.713            1.000   
Positive Feedback Count        0.009  0.013   0.026            0.026   

                         Positive Feedback Count  
Clothing ID                                0.009  
Age                                        0.013  
Rating                                     0.026  
Recommended IND                            0.026  
Positive Feedback Count                    1.000  


In [4]:
class DataCleaningPipeline:
    """
    Custom data cleaning pipeline that automatically detects, logs, and removes
    irrelevant/redundant columns and duplicate rows while protecting critical fields.
    """
    def __init__(
        self,
        protected_cols=None,
        near_dup_subset=None,
        variance_threshold=0.95,
        corr_threshold=0.90,
        id_uniqueness_threshold=0.98,
        drop_near_duplicates=False
    ):
        self.protected_cols = set(protected_cols or [])
        self.near_dup_subset = near_dup_subset
        self.variance_threshold = variance_threshold
        self.corr_threshold = corr_threshold
        self.id_uniqueness_threshold = id_uniqueness_threshold
        self.drop_near_duplicates = drop_near_duplicates
        self.audit_log = []

    def _log(self, stage, action, details):
        self.audit_log.append({'stage': stage, 'action': action, 'details': details})
        print(f"[{stage}] {action}: {details}")

    def remove_irrelevant_columns(self, df):
        df = df.copy()
        cols_to_drop = {}

        for col in df.columns:
            # 1. Auto-generated index columns
            if str(col).lower().startswith('unnamed:'):
                cols_to_drop[col] = "Auto-generated index column"
                continue

            top_ratio = df[col].value_counts(normalize=True, dropna=False).iloc[0]
            if df[col].nunique(dropna=False) <= 1:
                cols_to_drop[col] = "Zero-variance (constant) column"
                continue
            elif top_ratio >= self.variance_threshold:
                cols_to_drop[col] = f"Near-zero variance (top value = {top_ratio:.2%})"
                continue

            unique_ratio = df[col].nunique(dropna=True) / max(len(df), 1)
            if unique_ratio >= self.id_uniqueness_threshold and (
                'id' in col.lower() or df[col].dtype in ['int64', 'int32']
            ):
                cols_to_drop[col] = f"ID-like high-cardinality column (unique ratio = {unique_ratio:.2%})"

        num_cols = [c for c in df.select_dtypes(include=[np.number]).columns if c not in cols_to_drop]
        if len(num_cols) > 1:
            corr_matrix = df[num_cols].corr().abs()
            upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
            for col in upper.columns:
                high_corr = upper.index[upper[col] >= self.corr_threshold].tolist()
                if high_corr:
                    cols_to_drop[col] = f"Redundant numeric correlation (>= {self.corr_threshold}) with {high_corr}"

        final_drop = []
        for col, reason in cols_to_drop.items():
            if col in self.protected_cols:
                self._log("Columns", "PROTECTED", f"Retained '{col}' despite flag ({reason}).")
            else:
                final_drop.append(col)
                self._log("Columns", "REMOVED", f"Dropped '{col}' — {reason}.")

        if not final_drop:
            self._log("Columns", "CLEAN", "No irrelevant or redundant columns detected.")
        else:
            df = df.drop(columns=final_drop)

        return df

    def remove_duplicates(self, df):
        df = df.copy()

        exact_dups = int(df.duplicated().sum())
        if exact_dups > 0:
            df = df.drop_duplicates(keep='first').reset_index(drop=True)
            self._log("Duplicates", "REMOVED", f"{exact_dups} exact duplicate rows.")
        else:
            self._log("Duplicates", "CLEAN", "0 exact duplicate rows found.")

        if self.near_dup_subset:
            valid_subset = [c for c in self.near_dup_subset if c in df.columns]
            near_dups = int(df.duplicated(subset=valid_subset).sum())
            if near_dups > 0 and self.drop_near_duplicates:
                df = df.drop_duplicates(subset=valid_subset, keep='first').reset_index(drop=True)
                self._log("Near-Duplicates", "REMOVED", f"{near_dups} rows matching {valid_subset}.")
            elif near_dups > 0:
                self._log(
                    "Near-Duplicates",
                    "FLAGGED_ONLY",
                    f"{near_dups} rows share {valid_subset} (retained to avoid information loss)."
                )
            else:
                self._log("Near-Duplicates", "CLEAN", f"0 near-duplicates on {valid_subset}.")

        return df

    def run(self, df):
        self._log("Pipeline", "START", f"Initial shape: {df.shape}")
        df_clean = self.remove_irrelevant_columns(df)
        df_clean = self.remove_duplicates(df_clean)
        self._log("Pipeline", "END", f"Final shape: {df_clean.shape}")
        return df_clean

    def get_audit_report(self):
        return pd.DataFrame(self.audit_log)

In [5]:
pipeline = DataCleaningPipeline(
    protected_cols=['Clothing ID', 'Rating', 'Recommended IND', 'Review Text'],
    near_dup_subset=['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating'],
    variance_threshold=0.95,
    corr_threshold=0.90,
    drop_near_duplicates=False
)

df_clean = pipeline.run(df)
audit_df = pipeline.get_audit_report()
audit_df

[Pipeline] START: Initial shape: (1500, 10)
[Columns] CLEAN: No irrelevant or redundant columns detected.
[Duplicates] CLEAN: 0 exact duplicate rows found.
[Near-Duplicates] FLAGGED_ONLY: 17 rows share ['Clothing ID', 'Age', 'Title', 'Review Text', 'Rating'] (retained to avoid information loss).
[Pipeline] END: Final shape: (1500, 10)


,stage,action,details
0,Pipeline,START,"Initial shape: (1500, 10)"
1,Columns,CLEAN,No irrelevant or redundant columns detected.
2,Duplicates,CLEAN,0 exact duplicate rows found.
3,Near-Duplicates,FLAGGED_ONLY,"17 rows share ['Clothing ID', 'Age', 'Title', ..."
4,Pipeline,END,"Final shape: (1500, 10)"


In [6]:
df_test = df.copy()
df_test = pd.concat([df_test, df_test.iloc[:5]], ignore_index=True)  
df_test['Unnamed: 0'] = range(len(df_test))                          
df_test['Constant_Col'] = 'Fashion'                                  
df_test['Rating_Copy'] = df_test['Rating'] * 2                       

test_pipeline = DataCleaningPipeline(
    protected_cols=['Clothing ID', 'Rating', 'Recommended IND', 'Review Text'],
    variance_threshold=0.95,
    corr_threshold=0.90
)

df_test_clean = test_pipeline.run(df_test)
print("\nRestored exact original shape:", df_test_clean.shape == df.shape)

[Pipeline] START: Initial shape: (1505, 13)
[Columns] REMOVED: Dropped 'Unnamed: 0' — Auto-generated index column.
[Columns] REMOVED: Dropped 'Constant_Col' — Zero-variance (constant) column.
[Columns] REMOVED: Dropped 'Rating_Copy' — Redundant numeric correlation (>= 0.9) with ['Rating'].
[Duplicates] REMOVED: 5 exact duplicate rows.
[Pipeline] END: Final shape: (1500, 10)

Restored exact original shape: True


In [7]:
df_clean.to_csv('womens_clothing_cleaned.csv', index=False)
audit_df.to_csv('cleaning_audit_log.csv', index=False)
print("Exported 'womens_clothing_cleaned.csv' and 'cleaning_audit_log.csv'.")

Exported 'womens_clothing_cleaned.csv' and 'cleaning_audit_log.csv'.
